ADNI Documentation: https://adni.loni.usc.edu/quick-start-guide-asset/anatomy2.html

This notebook is investigating the data of the dataset. Mainly checking if there are images that belong to the same patient across the train/test splits.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd

In [2]:
DATASET_ROOT = Path('~/Documents/Datasets/ADNI/AD_NC').expanduser()
METADATA_PATH = Path('~/Documents/Datasets/ADNI/meta_data_with_label.json').expanduser()

# get paths of each folder in dataset
FOLDERS = [
    DATASET_ROOT / 'train' / 'AD',
    DATASET_ROOT / 'train' / 'NC',
    DATASET_ROOT / 'test' / 'AD',
    DATASET_ROOT / 'test' / 'NC',
]

In [3]:
# open and read meta data json file
with METADATA_PATH.open('r', encoding='utf-8') as file:
    metadata = json.load(file)

# regex pattern for matching patient id from meta data
PATIENT_PATTERN = re.compile(r"ADNI_(\d{3}_S_\d{4})", re.IGNORECASE)

# read patient id from raw key in meta data
def patient_id_from_record(record):
    value = record.get('raw')
    match = PATIENT_PATTERN.search(value)
    return match.group(1).upper()

# build dict of scan_id : patient_id
scan_to_patient = {
    str(scan_id): patient_id_from_record(record)
    for scan_id, record in metadata.items()
}

print(f'Metadata records: {len(metadata)}')
print(f'Scans mapped to a patient: {len(scan_to_patient):}')
print(f'Unique patients in metadata: {len(set(scan_to_patient.values()))}')

Metadata records: 2189
Scans mapped to a patient: 2189
Unique patients in metadata: 942


In [4]:
# get scan id from the file name of the image
def scan_id_from_filename(path):
    return path.name.split("_", 1)[0]

rows = []

# iterate through dataset folder
for folder_path in FOLDERS:
    split = folder_path.parts[-2] # train or test
    for image_path in sorted(folder_path.rglob('*')):
        scan_id = scan_id_from_filename(image_path)
        record = metadata[scan_id]

        # add details related to image to list
        rows.append({
            'split': split,
            'image_name': image_path.name,
            'scan_id': scan_id,
            'patient_id': scan_to_patient.get(scan_id),
        })

# convert list to pandas dataframe
dataset_df = pd.DataFrame(rows)

print(f'Images Recorded: {len(dataset_df)}')
display(dataset_df.head())

Images Recorded: 30521


,split,image_name,scan_id,patient_id
0,train,218391_78.jpeg,218391,037_S_4001
1,train,218391_79.jpeg,218391,037_S_4001
2,train,218391_80.jpeg,218391,037_S_4001
3,train,218391_81.jpeg,218391,037_S_4001
4,train,218391_82.jpeg,218391,037_S_4001


In [5]:
# get the unique patients in each split
train_patients = set(
    dataset_df.loc[dataset_df["split"] == "train", "patient_id"]
)
test_patients = set(
    dataset_df.loc[dataset_df["split"] == "test", "patient_id"]
)

overlapping_patients = train_patients & test_patients

print(f"Unique training patients: {len(train_patients)}")
print(f"Unique testing patients: {len(test_patients)}")
print(f"Patients in both splits: {len(overlapping_patients)}")

Unique training patients: 565
Unique testing patients: 331
Patients in both splits: 216


In [6]:
# get images that are overlapping
overlap_images = dataset_df[dataset_df["patient_id"].isin(overlapping_patients)].copy()

print(f"Number of images overlapping: {len(overlap_images)}")

Number of images overlapping: 14381


Nearly half of the images have the same patient but are overlapping across the train and test splits :(

To solve this I am going to combine the test and train splits and then manually split them myself by patients.

Ideally by doing this I should be able to achieve roughly 70% train, 20% validation, 10% test.